# Dataset Viewer

Interactive viewer for the Uncertainty Fusion outdoor dataset.

Select an **Event**, **Run**, **Category**, and **Modality**, then drag the **Frame** slider to scrub through frames.

| Modality | Description |
|----------|-------------|
| RGB | Raw camera frames (`aerial/rgb/` + `ground/rgb/`) |
| GT BBox | Ground-truth bounding-box overlays (`aerial/gt_bbox/` + `ground/gt_bbox/`) |

In [1]:
import io, json, math, requests, warnings
from functools import lru_cache
from pathlib import Path
from PIL import Image, ImageDraw
import ipywidgets as widgets
from IPython.display import display

warnings.filterwarnings('ignore')

_RESAMPLE    = getattr(Image, 'Resampling', Image).BILINEAR
DATASET_ROOT = Path('/Documents/dataset')
TARGET_HEIGHT = 400          # camera view height in px
MAP_PX        = TARGET_HEIGHT // 2   # GPS map panel size (square)
TILE_SIZE     = 256
TILE_URL      = 'https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}'
DETAIL_ZOOM   = 18           # zoom level used for the live satellite panel

In [2]:
# ── Dataset discovery ─────────────────────────────────────────────────────────
def get_events():
    return sorted([
        d.name.replace('-Processed-release', '')
        for d in DATASET_ROOT.iterdir()
        if d.is_dir() and d.name.endswith('-Processed-release')
    ])

def get_runs(event):
    d = DATASET_ROOT / f'{event}-Processed-release'
    if not d.exists():
        return []
    return sorted([x.name for x in d.iterdir() if x.is_dir() and x.name.startswith('run_')])

def get_categories(event, run):
    d = DATASET_ROOT / f'{event}-Processed-release' / run
    if not d.exists():
        return []
    return sorted([x.name for x in d.iterdir() if x.is_dir() and not x.name.startswith('.')])

def get_frame_files(event, run, category, platform, modality):
    d = DATASET_ROOT / f'{event}-Processed-release' / run / category / platform / modality
    if not d.exists():
        return []
    if modality == 'rgb':
        if platform == 'ground':
            # some events use stitched_*, others use rgb_*
            files = sorted(d.glob('stitched_*.png'))
            if not files:
                files = sorted(d.glob('rgb_*.png'))
            return files
        else:
            return sorted(d.glob('rgb_*.png'))
    else:
        return sorted(d.glob('gt_bbox_*.png'))


# ── Slippy-map tile math ───────────────────────────────────────────────────────
def _tile_xy(lat, lon, zoom):
    lat_r = math.radians(lat)
    n = 1 << zoom
    x = int((lon + 180.0) / 360.0 * n)
    y = int((1.0 - math.log(math.tan(lat_r) + 1.0 / math.cos(lat_r)) / math.pi) / 2.0 * n)
    return x, y

def _latlon_to_px(lat, lon, zoom):
    lat_r = math.radians(lat)
    n = 1 << zoom
    px = (lon + 180.0) / 360.0 * n * TILE_SIZE
    py = (1.0 - math.log(math.tan(lat_r) + 1.0 / math.cos(lat_r)) / math.pi) / 2.0 * n * TILE_SIZE
    return px, py

def _choose_zoom(lat_span, lon_span):
    for z in range(20, 10, -1):
        n = 1 << z
        if max(lat_span * n * TILE_SIZE / 180.0,
               lon_span * n * TILE_SIZE / 360.0) <= MAP_PX:
            return z
    return 14

@lru_cache(maxsize=1024)
def _fetch_tile(z, x, y):
    try:
        r = requests.get(TILE_URL.format(z=z, y=y, x=x), timeout=5,
                         headers={'User-Agent': 'DatasetViewer/1.0'})
        if r.status_code == 200:
            return Image.open(io.BytesIO(r.content)).convert('RGB')
    except Exception:
        pass
    return Image.new('RGB', (TILE_SIZE, TILE_SIZE), (60, 60, 60))


# ── GPS track cache ────────────────────────────────────────────────────────────
_gps_cache = {}

def _load_gps_track(event, run, category):
    key = (event, run, category)
    if key in _gps_cache:
        return _gps_cache[key]

    def _read(directory, pattern):
        coords = []
        for f in sorted(directory.glob(pattern)):
            try:
                d = json.loads(f.read_text())
                coords.append((d['latitude'], d['longitude']))
            except Exception:
                pass
        return coords

    a_dir = DATASET_ROOT / f'{event}-Processed-release' / run / category / 'aerial' / 'gps_position'
    g_dir = DATASET_ROOT / f'{event}-Processed-release' / run / category / 'ground' / 'gps_fix'
    a_coords = _read(a_dir, 'gps_position_*.json')
    g_coords = _read(g_dir, 'gps_fix_*.json')

    if not a_coords and not g_coords:
        _gps_cache[key] = None
        return None

    entry = dict(a_coords=a_coords, g_coords=g_coords)
    _gps_cache[key] = entry
    return entry


def _draw_gps_map(entry, a_idx, g_idx, alpha):
    """High-res satellite map centred on the current GPS midpoint."""
    a_coords = entry['a_coords']
    g_coords = entry['g_coords']

    # Centre on current vehicle positions
    pts = []
    if 0 <= a_idx < len(a_coords):
        pts.append(a_coords[a_idx])
    if 0 <= g_idx < len(g_coords):
        pts.append(g_coords[g_idx])
    if not pts:
        all_c = a_coords + g_coords
        clat = sum(p[0] for p in all_c) / len(all_c)
        clon = sum(p[1] for p in all_c) / len(all_c)
    else:
        clat = sum(p[0] for p in pts) / len(pts)
        clon = sum(p[1] for p in pts) / len(pts)

    zoom = DETAIL_ZOOM
    tx_c, ty_c = _tile_xy(clat, clon, zoom)

    # Fetch a 3×3 tile grid centred on the tile containing clat/clon
    radius = 1
    mosaic = Image.new('RGB', ((2 * radius + 1) * TILE_SIZE, (2 * radius + 1) * TILE_SIZE))
    for dy in range(-radius, radius + 1):
        for dx in range(-radius, radius + 1):
            tile = _fetch_tile(zoom, tx_c + dx, ty_c + dy)
            mosaic.paste(tile, ((dx + radius) * TILE_SIZE, (dy + radius) * TILE_SIZE))

    # Pixel origin of the mosaic in global tile space
    ox = (tx_c - radius) * TILE_SIZE
    oy = (ty_c - radius) * TILE_SIZE

    # Crop MAP_PX × MAP_PX centred on clat/clon
    cpx, cpy = _latlon_to_px(clat, clon, zoom)
    half = MAP_PX // 2
    left = int(max(0, min(cpx - ox - half, mosaic.width  - MAP_PX)))
    top  = int(max(0, min(cpy - oy - half, mosaic.height - MAP_PX)))

    sat = mosaic.crop((left, top, left + MAP_PX, top + MAP_PX))

    # Alpha-blend with black
    black = Image.new('RGB', (MAP_PX, MAP_PX), (0, 0, 0))
    sat   = Image.blend(black, sat, max(0.0, min(1.0, alpha)))
    draw  = ImageDraw.Draw(sat)

    def to_map(lat, lon):
        px, py = _latlon_to_px(lat, lon, zoom)
        return int(px - ox - left), int(py - oy - top)

    # Track dots (subsampled, clip to visible area)
    for lat, lon in a_coords[::2]:
        mx, my = to_map(lat, lon)
        if 0 <= mx < MAP_PX and 0 <= my < MAP_PX:
            draw.ellipse([mx-2, my-2, mx+2, my+2], fill=(248, 112, 39))
    for lat, lon in g_coords[::2]:
        mx, my = to_map(lat, lon)
        if 0 <= mx < MAP_PX and 0 <= my < MAP_PX:
            draw.ellipse([mx-2, my-2, mx+2, my+2], fill=(246, 182, 12))

    # Current-position markers
    for idx, coords, color in [(a_idx, a_coords, (248, 112, 39)),
                                (g_idx, g_coords, (246, 182, 12))]:
        if 0 <= idx < len(coords):
            mx, my = to_map(*coords[idx])
            draw.ellipse([mx-6, my-6, mx+6, my+6], fill=color,
                         outline=(255, 255, 255), width=2)

    return sat

In [3]:
# ── Bootstrap ────────────────────────────────────────────────────────────────
_events = get_events()
_runs0  = get_runs(_events[0])
_cats0  = get_categories(_events[0], _runs0[0])

# ── Widgets ──────────────────────────────────────────────────────────────────
w_event = widgets.Dropdown(
    options=_events, value=_events[0], description='Event:',
    layout=widgets.Layout(width='200px')
)
w_run = widgets.Dropdown(
    options=_runs0, value=_runs0[0], description='Run:',
    layout=widgets.Layout(width='200px')
)
w_category = widgets.Dropdown(
    options=_cats0, value=_cats0[0], description='Category:',
    layout=widgets.Layout(width='230px')
)
w_modality = widgets.ToggleButtons(
    options=['RGB', 'GT BBox'], value='RGB', description='Modality:',
    style={'button_width': '100px'}
)
w_frame = widgets.IntSlider(
    min=0, max=0, step=1, value=0, description='Frame:',
    continuous_update=True, layout=widgets.Layout(width='80%')
)
w_label = widgets.HTML(value='')
w_img   = widgets.Image(format='jpeg', layout=widgets.Layout(width='100%'))

_busy = False


# ── Helpers ───────────────────────────────────────────────────────────────────
def _update_frame_range():
    mod_dir = 'rgb' if w_modality.value == 'RGB' else 'gt_bbox'
    af = get_frame_files(w_event.value, w_run.value or '', w_category.value or '', 'aerial', mod_dir)
    gf = get_frame_files(w_event.value, w_run.value or '', w_category.value or '', 'ground', mod_dir)
    n = min(len(af), len(gf))
    w_frame.max   = max(0, n - 1)
    w_frame.value = 0


def on_selection_change(change=None):
    global _busy
    if _busy:
        return
    _busy = True
    try:
        event = w_event.value
        runs  = get_runs(event)
        if list(w_run.options) != runs:
            w_run.options = runs
        if w_run.value not in runs:
            w_run.value = runs[0] if runs else None
        run  = w_run.value or ''
        cats = get_categories(event, run)
        if list(w_category.options) != cats:
            w_category.options = cats
        if w_category.value not in cats:
            w_category.value = cats[0] if cats else None
        _update_frame_range()
    finally:
        _busy = False
    show_frame()


def show_frame(change=None):
    if _busy:
        return
    event    = w_event.value
    run      = w_run.value or ''
    category = w_category.value or ''
    modality = w_modality.value
    mod_dir  = 'rgb' if modality == 'RGB' else 'gt_bbox'
    idx      = w_frame.value

    af = get_frame_files(event, run, category, 'aerial', mod_dir)
    gf = get_frame_files(event, run, category, 'ground', mod_dir)
    n  = min(len(af), len(gf))
    if n == 0:
        w_label.value = '<i>No frames found for this selection.</i>'
        return

    idx = min(idx, n - 1)

    # Camera images
    a = Image.open(af[idx]).convert('RGB')
    g = Image.open(gf[idx]).convert('RGB')
    a_w = max(1, round(a.width * TARGET_HEIGHT / a.height))
    g_w = max(1, round(g.width * TARGET_HEIGHT / g.height))
    a = a.resize((a_w, TARGET_HEIGHT), _RESAMPLE)
    g = g.resize((g_w, TARGET_HEIGHT), _RESAMPLE)

    # Composite aerial + ground flush (no gap)
    canvas = Image.new('RGB', (a_w + g_w, TARGET_HEIGHT), (20, 20, 20))
    canvas.paste(a, (0, 0))
    canvas.paste(g, (a_w, 0))

    # GPS map overlaid at the seam, vertically centered
    gps = _load_gps_track(event, run, category)
    if gps:
        map_img = _draw_gps_map(gps, idx, idx, 0.85)
        map_x = max(0, min(a_w - MAP_PX // 2, canvas.width - MAP_PX))
        map_y = (TARGET_HEIGHT - MAP_PX) // 2
        # canvas.paste(map_img, (map_x, map_y **2))
        canvas.paste(map_img, (0, 0))

    buf = io.BytesIO()
    canvas.save(buf, format='JPEG', quality=85)
    w_img.value = buf.getvalue()

    w_label.value = (
        f'<b>{event} | {run} | {category} | {modality} | Frame {idx + 1} / {n}</b>'
        f'&emsp;<span style="color:#f87027">&#9679; Aerial</span>'
        f'&emsp;<span style="color:#f6b60c">&#9679; Ground</span>'
    )


# ── Wire observers ────────────────────────────────────────────────────────────
w_event.observe(on_selection_change, names='value')
w_run.observe(on_selection_change, names='value')
w_category.observe(on_selection_change, names='value')
w_modality.observe(on_selection_change, names='value')
w_frame.observe(show_frame, names='value')

on_selection_change()

display(widgets.VBox([
    widgets.HBox([w_event, w_run, w_category]),
    w_modality,
    w_frame,
    w_label,
    w_img,
]))